# 実験1：二段階モデルの第1段階「取りこぼし」分析

## 目的
現在の二段階着雪量予測について、**第1段階で真の着雪ありを落としていることがWMAEにどれだけ効いているか**を確認する。

確認するもの：
1. `実測 > 0 & 第1段階flag = 0` の取りこぼし
2. `第1段階flag = 1` を通過した後の第2段階予測誤差
3. 第1段階の確率が保存されている場合の閾値感度

Accuracyではなく、最終評価指標であるWMAEへの寄与を基準に判断する。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/home/keiseki/JR_train_snow")
if not (ROOT / "30.src").exists():
    ROOT = Path.cwd()

SRC_ROOT = ROOT / "30.src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("ROOT:", ROOT)


In [ ]:
# 設定
CONFIG_PATH = ROOT / "00.config" / "config.yaml"
PATH_CONFIG_PATH = ROOT / "00.config" / "path.yaml"

from jr_snow.config import build_feature_columns, load_project_config
from jr_snow.data import load_train_test_data

config, path_config = load_project_config(str(CONFIG_PATH), str(PATH_CONFIG_PATH))
feature_columns = build_feature_columns(config)
path_map = path_config.get("INTERIUM_PATH", path_config)

print("split_date :", config.get("split_date"))
print("thresholds :", config.get("thresholds"))
print("train_data :", path_map.get("train_data"))
print("test_data  :", path_map.get("test_data"))


In [ ]:
# 学習データ読み込み
train_df, test_df = load_train_test_data(
    train_data_path=path_map.get("train_data"),
    test_data_path=path_map.get("test_data"),
    path_config=path_config,
)

print("train:", train_df.shape)
print("test :", test_df.shape)


## 二段階モデルの検証結果を読み込む

下の `RESULT_PATH` を、現在の二段階モデルで作った**検証データの予測結果CSV**に変更する。

最低限必要な列：
- `年月日`
- `列車番号`
- `合計`
- `着雪量予測フラグ`
- `予測`

第1段階の確率を保存している場合は、さらに `着雪確率` を入れると閾値実験もできる。

In [ ]:
# ★ここだけ現在の結果ファイルに変更
RESULT_PATH = ROOT / "artifacts" / "two_stage_validation_predictions.csv"

COLUMN_MAP = {
    "date": "年月日",
    "train_no": "列車番号",
    "target": "合計",
    "flag": "着雪量予測フラグ",
    "prediction": "予測",
}

if RESULT_PATH.exists():
    result_df = pd.read_csv(RESULT_PATH)
    print("読み込み:", RESULT_PATH)
    print("shape:", result_df.shape)
    display(result_df.head())
else:
    print("結果ファイルがありません。")
    print("RESULT_PATHを現在の二段階モデルの検証予測CSVに変更してください。")


In [ ]:
# 列チェック・WMAE計算用列を作成
if "result_df" in globals():
    missing = [v for v in COLUMN_MAP.values() if v not in result_df.columns]
    print("不足列:", missing)

    if not missing:
        error_df = result_df.copy()
        error_df["年月日"] = pd.to_datetime(error_df[COLUMN_MAP["date"]])
        error_df["実測"] = pd.to_numeric(error_df[COLUMN_MAP["target"]], errors="coerce")
        error_df["flag"] = pd.to_numeric(
            error_df[COLUMN_MAP["flag"]], errors="coerce"
        ).fillna(0).astype(int)
        error_df["予測"] = pd.to_numeric(
            error_df[COLUMN_MAP["prediction"]], errors="coerce"
        )

        error_df["絶対誤差"] = (error_df["予測"] - error_df["実測"]).abs()
        error_df["WMAE重み"] = 1 + 10000 * error_df["実測"].abs()
        error_df["WMAE寄与"] = error_df["絶対誤差"] * error_df["WMAE重み"]

        error_df["実測着雪あり"] = (error_df["実測"] > 0).astype(int)
        error_df["第1段階取りこぼし"] = (
            (error_df["実測"] > 0) & (error_df["flag"] == 0)
        )
        error_df["第2段階評価対象"] = (
            (error_df["実測"] > 0) & (error_df["flag"] == 1)
        )

        print("rows:", len(error_df))
        display(error_df.head())


## 1. WMAEの分解

ここが実験の中心。

- 第1段階で落とされた着雪あり
- 第1段階を通過した着雪あり
- 実測0

のWMAE寄与を比較する。

In [ ]:
if "error_df" in globals():
    total_contribution = error_df["WMAE寄与"].sum()

    rows = []
    masks = {
        "全体": np.ones(len(error_df), dtype=bool),
        "第1段階取りこぼし": error_df["第1段階取りこぼし"],
        "第2段階評価対象": error_df["第2段階評価対象"],
        "実測0": error_df["実測"] <= 0,
    }

    for name, mask in masks.items():
        contribution = error_df.loc[mask, "WMAE寄与"].sum()
        rows.append({
            "区分": name,
            "件数": int(mask.sum()),
            "WMAE寄与": contribution,
            "全体への寄与率": contribution / total_contribution if total_contribution else np.nan,
        })

    summary_df = pd.DataFrame(rows)
    display(summary_df)

    print("全体WMAE:", total_contribution / len(error_df))


In [ ]:
# 第1段階取りこぼしの上位ケース
if "error_df" in globals():
    miss_df = error_df[error_df["第1段階取りこぼし"]].sort_values(
        "WMAE寄与", ascending=False
    )

    print("取りこぼし件数:", len(miss_df))

    display(
        miss_df[
            [COLUMN_MAP["date"], COLUMN_MAP["train_no"],
             "実測", "予測", "flag", "絶対誤差", "WMAE寄与"]
        ].head(30)
    )


In [ ]:
# 着雪ありの取りこぼし率
if "error_df" in globals():
    snow_mask = error_df["実測"] > 0
    miss_mask = error_df["第1段階取りこぼし"]

    snow_count = snow_mask.sum()
    miss_count = miss_mask.sum()

    print("実測着雪あり:", snow_count)
    print("第1段階取りこぼし:", miss_count)
    print("着雪あり取りこぼし率:", miss_count / snow_count if snow_count else np.nan)


## 2. 第2段階だけの性能

`flag=1` を通過したデータについて、着雪量予測そのものの誤差を確認する。

ここで第2段階の誤差が大きければ、第1段階よりも第2段階モデルを改善するべき。

In [ ]:
if "error_df" in globals():
    stage2_df = error_df[error_df["flag"] == 1].copy()

    stage2_wmae = (
        stage2_df["WMAE寄与"].sum() / len(stage2_df)
        if len(stage2_df) else np.nan
    )

    print("第2段階通過件数:", len(stage2_df))
    print("第2段階通過データのWMAE:", stage2_wmae)

    display(
        stage2_df.sort_values("WMAE寄与", ascending=False)[
            [COLUMN_MAP["date"], COLUMN_MAP["train_no"],
             "実測", "予測", "flag", "絶対誤差", "WMAE寄与"]
        ].head(30)
    )


## 3. 第1段階閾値の感度分析

第1段階の**確率**が結果CSVに保存されている場合のみ実行する。

閾値を変えた仮想flagを作り、最終WMAEを比較する。

重要なのは分類Accuracyではなく、**最終WMAEが最小になる閾値**。

In [ ]:
PROB_COLUMN = "着雪確率"
THRESHOLDS = [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.015, 0.020]

if "error_df" in globals() and PROB_COLUMN in error_df.columns:
    prob = pd.to_numeric(error_df[PROB_COLUMN], errors="coerce")

    results = []

    for threshold in THRESHOLDS:
        flag = (prob >= threshold).astype(int)
        gated_pred = np.where(flag == 1, error_df["予測"], 0.0)

        abs_error = np.abs(gated_pred - error_df["実測"])
        contribution = abs_error * (1 + 10000 * np.abs(error_df["実測"]))

        snow_count = (error_df["実測"] > 0).sum()
        missed = ((error_df["実測"] > 0) & (flag == 0)).sum()

        results.append({
            "threshold": threshold,
            "WMAE": contribution.mean(),
            "取りこぼし件数": missed,
            "取りこぼし率": missed / snow_count if snow_count else np.nan,
            "flag=1件数": int(flag.sum()),
        })

    threshold_df = pd.DataFrame(results)
    display(threshold_df)

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_df["threshold"], threshold_df["WMAE"], marker="o")
    plt.xlabel("Stage-1 threshold")
    plt.ylabel("WMAE")
    plt.title("Stage-1 threshold vs WMAE")
    plt.grid(True)
    plt.show()
else:
    print(f"{PROB_COLUMN} が結果CSVにないため、閾値分析はスキップします。")


# 実験1の判定

### A. 第1段階取りこぼしのWMAE寄与が大きい
→ **第1段階の改善を優先**

- 閾値を下げる
- 第1段階の特徴量を改善
- 降雪イベントの継続時間などを追加

### B. 第1段階取りこぼしは小さく、第2段階の誤差が大きい
→ **第2段階の着雪量予測を改善**

- `合計 > 0` の学習条件
- 降雪量・気温・降水継続時間
- 直近3h/6h/12hの気象量

### C. 閾値を少し変えるだけでWMAEが大きく変わる
→ **第1段階の閾値最適化が有力**

最終的に見る数字は、

1. 第1段階取りこぼしのWMAE寄与率
2. 着雪あり取りこぼし率
3. 第2段階のWMAE
4. 閾値別WMAE

の4つ。